In [0]:
from pyspark.sql import functions as F
CATALOG = "workspace"

In [0]:
silver_orders_t = spark.table(f"{CATALOG}.silver.orders")

# one existing order with a corrected quantity...
changed = (silver_orders_t.limit(1)
           .withColumn("quantity", F.when(F.col("quantity") == 5, 4).otherwise(5)))

# ...and one brand-new order
new = spark.sql("""SELECT 'MERGE-0001' AS order_id, DATE'2026-10-22' AS order_date,
                          1001 AS customer_id, 'Merge Test' AS customer_name, 'Mumbai' AS city,
                          1 AS product_id, 2 AS quantity, 'UPI' AS payment_type""")

updates = changed.unionByName(new)
updates.createOrReplaceTempView("order_updates")
display(updates)

version_before = (spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.orders")
                  .agg(F.max("version")).first()[0])
rows_before = silver_orders_t.count()
print("version before:", version_before, "| rows before:", rows_before)

order_id,order_date,customer_id,customer_name,city,product_id,quantity,payment_type
20260923-00000,2026-09-23,1414,Aryan Maharaj,Mumbai,6,5,COD
MERGE-0001,2026-10-22,1001,Merge Test,Mumbai,1,2,UPI


version before: 3 | rows before: 15500


In [0]:
spark.sql(f"""
  MERGE INTO {CATALOG}.silver.orders AS t
  USING order_updates AS s
  ON t.order_id = s.order_id
  WHEN MATCHED AND t.quantity <> s.quantity THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")

print("rows after:", spark.table(f"{CATALOG}.silver.orders").count())
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.orders")
        .select("version", "operation", "operationMetrics").limit(3))

rows after: 15501


version,operation,operationMetrics
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 6242, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 8891, materializeSourceTimeMs -> 796, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2116, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 5822)"
3,RESTORE,"Map(numRestoredFiles -> 1, removedFilesSize -> 142468, numRemovedFiles -> 3, restoredFilesSize -> 136226, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 136226)"
2,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 6242, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6102, materializeSourceTimeMs -> 833, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2237, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2909)"


In [0]:
spark.sql(f"""
  MERGE INTO {CATALOG}.silver.orders AS t
  USING order_updates AS s
  ON t.order_id = s.order_id
  WHEN MATCHED AND t.quantity <> s.quantity THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")

print("rows after:", spark.table(f"{CATALOG}.silver.orders").count())
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.orders")
        .select("version", "operation", "operationMetrics").limit(3))

rows after: 15501


version,operation,operationMetrics
5,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3126, numTargetBytesRemoved -> 3083, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3259, materializeSourceTimeMs -> 689, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 924, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1604)"
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 6242, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 8891, materializeSourceTimeMs -> 796, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2116, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 5822)"
3,RESTORE,"Map(numRestoredFiles -> 1, removedFilesSize -> 142468, numRemovedFiles -> 3, restoredFilesSize -> 136226, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 136226)"
